<a href="https://colab.research.google.com/github/mdNotFound/100-days-of-ai-engineering/blob/main/DAY_7_Transfer_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Transfer Learning

In [1]:
import torch
import torch.nn as nn
import torchvision
from torchvision import datasets
from torchvision import transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import os  ##for working with folders and file paths
import zipfile ##for unzipping the downloaded dataset
import requests ##for downloading a file from the internet
from pathlib import Path ##a nicer way to handle file paths than plain strings

print("Pytorch version: ",torch.__version__)
print("Torchvision version: ",torchvision.__version__)
print("Is a GPU available: ", torch.cuda.is_available())

Pytorch version:  2.11.0+cu128
Torchvision version:  0.26.0+cu128
Is a GPU available:  True


In [2]:
if torch.cuda.is_available():
  device="cuda"
else:
  device="cpu"
print(device)

cuda


In [3]:
data_path=Path("data")
image_path=data_path/"pizza_steak_sushi"
if image_path.is_dir():
  print("Data already exists at:",image_path)
else:
  print("Downloading data...")
  image_path.mkdir(parents=True,exist_ok=True) ## creating the folders

  url="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip"
  response=requests.get(url) ## downloading the zip into memory
  zip_file_path=data_path/"food.zip"
  with open(zip_file_path,"wb") as f: ##save it to disk
    f.write(response.content)
  with zipfile.ZipFile(zip_file_path, "r") as zip_ref:
    zip_ref.extractall(image_path) ##unzip into our folder
  os.remove(zip_file_path) ## delete the zip,we dont need it
  print("Done. Data is at:",image_path)

Done. Data is at: data/pizza_steak_sushi


In [4]:
for current_folder,subfolder,filename in os.walk(image_path):
  print(f"length of files: {len(filename)}")

length of files: 0
length of files: 0
length of files: 19
length of files: 31
length of files: 25
length of files: 0
length of files: 75
length of files: 72
length of files: 78


In [5]:
imagenet_mean=[0.485,0.456,0.406]
imagenet_std=[0.229,0.224,0.225]

train_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean,
                         std=imagenet_std)
])

test_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean,
                         std=imagenet_std)
])

print("Train transform has",len(train_transform.transforms),"steps")
print("Test transform has",len(test_transform.transforms),"steps")

Train transform has 4 steps
Test transform has 4 steps


In [6]:
train_dir=image_path/"train" ##data/pizza_steak_sushi/train
test_dir=image_path/"test" ##data/pizza_steak_sushi/test
##datasets know how to load one image at a time
train_dataset=datasets.ImageFolder(root=train_dir,transform=train_transform)
test_dataset=datasets.ImageFolder(root=test_dir,transform=test_transform)
##what classes did ImageFolder discover
class_names=train_dataset.classes
print("Classes found:",class_names)
print("Class -> index mapping:",train_dataset.class_to_idx)
print("Number of training images:",len(train_dataset))
print("Number of testing images: ",len(test_dataset))

Classes found: ['pizza', 'steak', 'sushi']
Class -> index mapping: {'pizza': 0, 'steak': 1, 'sushi': 2}
Number of training images: 225
Number of testing images:  75


In [7]:
BATCH_SIZE=32
train_dataloader=DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)
test_dataloader=DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
    )
print("Number of training batches:",len(train_dataloader))
print("Number of testing batches:",len(test_dataloader))

Number of training batches: 8
Number of testing batches: 3


In [8]:
#get one batch out of the dataloader
image_batch,label_batch = next(iter(train_dataloader))
print("Image batch shape:", image_batch.shape)
print("Label batch shape:",label_batch.shape)
print("Image dtype:",image_batch.dtype)
print("Pixel value range: min=",image_batch.min().item(),"max =",image_batch.max().item())
print("The 5 first labels:",label_batch[:5])

Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Image dtype: torch.float32
Pixel value range: min= -2.1179039478302 max = 2.640000104904175
The 5 first labels: tensor([0, 1, 0, 0, 0])


In [9]:
weights=torchvision.models.EfficientNet_B0_Weights.DEFAULT ## default gives the latest weights
model=torchvision.models.efficientnet_b0(weights=weights)
model=model.to(device)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 191MB/s]


In [13]:
for name,modules__ in model.named_children():
  print(name,modules__)

features Sequential(
  (0): Conv2dNormActivation(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): SiLU(inplace=True)
  )
  (1): Sequential(
    (0): MBConv(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): SiLU(inplace=True)
        )
        (1): SqueezeExcitation(
          (avgpool): AdaptiveAvgPool2d(output_size=1)
          (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
          (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
          (activation): SiLU(inplace=True)
          (scale_activation): Sigmoid()
        )
        (2): Conv2dNormActivation(
          (0): Conv2d(32, 16, kernel_size=(1, 1), stride

### Freeze the weights

In [15]:
##total trainable parameters available in my model
trainable_before=0
for parameter in model.parameters():
  if parameter.requires_grad==True:
    trainable_before+=parameter.numel()
print(trainable_before)

5288548


In [16]:
for parameter in model.features.parameters():
  parameter.requires_grad=False

In [18]:
trainable_after=0
for parameter in model.parameters():
  if parameter.requires_grad==True:
    trainable_after+=parameter.numel()
print(trainable_after)

1281000


In [20]:
model.classifier=nn.Sequential(
    nn.Dropout(p=0.2,inplace=True),
    nn.Linear(in_features=1280,out_features=3)
).to(device)

In [21]:
loss_function=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(params=model.parameters(),
                           lr=0.001)
print("Loss function:",loss_function)
print("Optimizer:",optimizer)

Loss function: CrossEntropyLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [22]:
def train_one_epoch(model,dataloader,loss_function,optimizer,device):
  model.train()
  total_loss=0
  total_correct=0
  total_samples=0
  for X,y in dataloader:
    X=X.to(device) ##[32,3,224,224]
    y=y.to(device) ## [32]
    y_pred_logits=model(X)
    loss=loss_function(y_pred_logits,y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss=total_loss+loss.item()
    y_pred_labels=torch.argmax(y_pred_logits,dim=1)
    total_correct=total_correct+(y_pred_labels==y).sum().item()
    total_samples=total_samples+len(y)
  average_loss=total_loss/len(dataloader)
  accuracy=total_correct/total_samples
  return average_loss,accuracy

In [33]:
def test_one_epoch(model,dataloader,loss_function,device):
  model.eval()
  total_loss=0
  total_correct=0
  total_samples=0
  with torch.no_grad():
    for X,y in dataloader:
      X=X.to(device) ##[32,3,224,224]
      y=y.to(device) ## [32]
      y_pred_logits=model(X)
      loss=loss_function(y_pred_logits,y)

      total_loss=total_loss+loss.item()
      y_pred_labels=torch.argmax(y_pred_logits,dim=1)
      total_correct=total_correct+(y_pred_labels==y).sum().item()
      total_samples=total_samples+len(y)
  average_loss=total_loss/len(dataloader)
  accuracy=total_correct/total_samples
  return average_loss,accuracy

In [34]:
NUMBER_OF_EPOCHS=5
train_loss_history=[]
train_acc_history=[]
test_loss_history=[]
test_acc_history=[]

for epoch in range(NUMBER_OF_EPOCHS):
  train_loss,train_acc=train_one_epoch(model,train_dataloader,loss_function,optimizer,device)
  test_loss,test_acc=test_one_epoch(model,test_dataloader,loss_function,device)
  train_loss_history.append(train_loss)
  train_acc_history.append(train_acc)
  test_loss_history.append(test_loss)
  test_acc_history.append(test_acc)
  print(f"Epoch {epoch+1}/{NUMBER_OF_EPOCHS} |"
        f"train_loss: {train_loss:.4f} | train_acc: {train_acc:.4f} |"
        f"test_loss: {test_loss:.4f} | test_acc: {test_acc:.4f}")
print("\nTraining finished!")

Epoch 1/5 |train_loss: 0.4183 | train_acc: 0.9378 |test_loss: 0.4282 | test_acc: 0.9067
Epoch 2/5 |train_loss: 0.4038 | train_acc: 0.9378 |test_loss: 0.4270 | test_acc: 0.9467
Epoch 3/5 |train_loss: 0.4954 | train_acc: 0.9200 |test_loss: 0.4380 | test_acc: 0.9067
Epoch 4/5 |train_loss: 0.4201 | train_acc: 0.9156 |test_loss: 0.3582 | test_acc: 0.8800
Epoch 5/5 |train_loss: 0.4055 | train_acc: 0.9244 |test_loss: 0.3756 | test_acc: 0.8800

Training finished!
